# 01 - Text Pipeline: Preprocessing & Baseline Experiments

**Stage:** Text-based fake news detection — exploration phase

**What this notebook does:**
- Converts cleaned news text into embeddings using Sentence-BERT
- Trains and compares baseline classifiers: Logistic Regression, XGBoost, and an MLP (Feedforward NN)
- Runs iterative XLM-RoBERTa fine-tuning experiments (multiple hyperparameter tries) to establish the best configuration

**Input:** cleaned/labeled text dataset (see `data/sample/`)
**Output:** baseline metrics + the best hyperparameter configuration, used as the starting point for `02_text_xlm_roberta_final_pipeline.ipynb`

> Part of the WAVE project — AI section (text-based fake news detection).

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
# --- Environment configuration ---
# Set BASE_DIR to wherever your data/model checkpoints live locally.
# If running on Google Colab, uncomment the two lines below.
# from google.colab import drive
# drive.mount('/content/drive')

import os
BASE_DIR = os.environ.get("WAVE_DATA_DIR", "./data")


In [ ]:
import torch


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


In [ ]:
import pandas as pd


df = pd.read_csv('Cleaned_Data_With_Preprocessed_Text.csv')
df

In [ ]:

label_counts = df['label'].value_counts()

print("🧾 Number of samples per label:")
print(label_counts)


import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.pie(label_counts, labels=label_counts.index, autopct='%1.1f%%', startangle=90, colors=['green', 'red'])
plt.title('Distribution of News Labels (Real vs Fake)')
plt.axis('equal')
plt.show()


# **تحويل النصوص إلى تمثيلات رقمية (BERT Embeddings)**

In [ ]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer
import torch
from tqdm import tqdm


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32



df = df[df['Clean_Arabic_Text'].notnull() & (df['Clean_Arabic_Text'].str.strip() != "")].reset_index(drop=True)


print("🔠 Loading English model...")
english_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=DEVICE)


print(" Loading Arabic BERT model...")
arabic_model_name = "asafaya/bert-base-arabic"
arabic_tokenizer = AutoTokenizer.from_pretrained(arabic_model_name)
arabic_model = AutoModel.from_pretrained(arabic_model_name).to(DEVICE)
arabic_model.eval()


def get_arabic_embeddings(texts):
    embeddings = []
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="Encoding Arabic Texts"):
        batch = texts[i:i+BATCH_SIZE]
        inputs = arabic_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=128)
        inputs = {key: val.to(DEVICE) for key, val in inputs.items()}
        with torch.no_grad():
            outputs = arabic_model(**inputs)
            batch_embeds = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings.extend(batch_embeds)
    return np.array(embeddings)


print(" Encoding English texts...")
english_embeddings = english_model.encode(df['Clean_English_Text'].tolist(), batch_size=BATCH_SIZE, show_progress_bar=True)


print(" Encoding Arabic texts...")
arabic_embeddings = get_arabic_embeddings(df['Clean_Arabic_Text'].tolist())


print(" Concatenating English + Arabic embeddings...")
combined_embeddings = np.concatenate([english_embeddings, arabic_embeddings], axis=1)


np.save("X_embeddings.npy", combined_embeddings)
df['label'].to_csv("y_labels.csv", index=False)

print(" Embedding generation complete. Files saved: X_embeddings.npy, y_labels.csv")


# **ML Section (LR,XGBOOST)**

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder


In [ ]:

X = np.load("X_embeddings.npy")
y = pd.read_csv("y_labels.csv").values.ravel()


label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


In [ ]:
# ========== تدريب نموذج Logistic Regression ==========
print("\n Training Logistic Regression...")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]


In [ ]:
# ========== تدريب نموذج XGBoost ==========
print("\n Training XGBoost...")
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]


In [ ]:
metrics_dict = {}

def evaluate_model(name, y_true, y_pred, y_prob):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred)
    auc  = roc_auc_score(y_true, y_prob)

    print(f"\n {name} Evaluation:")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")


    metrics_dict[name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1 Score': f1,
        'AUC': auc
    }


    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix - {name}")
    plt.grid(False)
    plt.show()


    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.2f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {name}")
    plt.legend(loc="lower right")
    plt.grid()
    plt.show()

evaluate_model("Logistic Regression", y_test, lr_preds, lr_probs)
evaluate_model("XGBoost", y_test, xgb_preds, xgb_probs)

# ========== رسم مقارنة شاملة للمقاييس ==========
metrics_df = pd.DataFrame(metrics_dict).T 
metrics_df.plot(kind='bar', figsize=(10, 6))
plt.title("Model Evaluation Metrics")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.grid(axis='y')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# **Neural Network Section**

# **Multilayer Perceptron (MLP) || Feedforward Neural Networks (FFNN)**

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import matplotlib.pyplot as plt

In [ ]:

X = np.load("X_embeddings.npy")
y = pd.read_csv("y_labels.csv").values.ravel()

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)


X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)


X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val_tensor   = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)


class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super(SimpleNN, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


input_dim = X_train.shape[1]
model = SimpleNN(input_dim)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


train_losses = []
val_losses = []


epochs = 30
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)
    loss.backward()
    optimizer.step()
    train_losses.append(loss.item())

  
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor).item()
        val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {loss.item():.4f} - Val Loss: {val_loss:.4f}")


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:

model.eval()
with torch.no_grad():
    y_pred_probs = model(X_val_tensor).numpy().flatten()
    y_preds = (y_pred_probs >= 0.5).astype(int)


acc = accuracy_score(y_val, y_preds)
prec = precision_score(y_val, y_preds)
rec = recall_score(y_val, y_preds)
f1 = f1_score(y_val, y_preds)
auc = roc_auc_score(y_val, y_pred_probs)

print("\n Neural Network Evaluation on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")


cm = confusion_matrix(y_val, y_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoder.classes_)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Neural Network")
plt.grid(False)
plt.show()


fpr, tpr, _ = roc_curve(y_val, y_pred_probs)
plt.figure()
plt.plot(fpr, tpr, label=f"Neural Net (AUC = {auc:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Neural Network")
plt.legend(loc="lower right")
plt.grid()
plt.show()

# **Fine Tuning**
# **إعداد البيانات لـ Fine-Tuning باستخدام XLM-RoBERTa**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


df = pd.read_csv("BASE_DIR/Senior Data/Cleaned_Data_With_Preprocessed_Text.csv")


df["text"] = df["Clean_English_Text"].fillna("") + " " + df["Clean_Arabic_Text"].fillna("")


df["label"] = df["label"].map({"real": 1, "fake": 0})


df = df[["text", "label"]].dropna().reset_index(drop=True)


train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)


train_df.to_csv("BASE_DIR/Senior Data/train.csv", index=False)
val_df.to_csv("BASE_DIR/Senior Data/val.csv", index=False)
test_df.to_csv("BASE_DIR/Senior Data/test.csv", index=False)

print("✅ Done! Files saved: train.csv, val.csv, test.csv")


In [ ]:
import pandas as pd


train_df = pd.read_csv("BASE_DIR/Senior Data/train.csv")
val_df = pd.read_csv("BASE_DIR/Senior Data/val.csv")
test_df = pd.read_csv("BASE_DIR/Senior Data/test.csv")


def summarize(df, name):
    print(f"\n {name} Set")
    print(f"عدد السطور: {len(df)}")
    print("عدد العينات لكل فئة:")
    print(df['label'].value_counts())


summarize(train_df, "Train")
summarize(val_df, "Validation")
summarize(test_df, "Test")


In [ ]:
!pip install -q transformers datasets evaluate scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import evaluate
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import matplotlib.pyplot as plt
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
)

In [ ]:

train_df = pd.read_csv("BASE_DIR/Senior Data/train.csv")
val_df = pd.read_csv("BASE_DIR/Senior Data/val.csv")
test_df = pd.read_csv("BASE_DIR/Senior Data/test.csv")


In [ ]:

ds = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

In [ ]:

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:

def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)


In [ ]:

tokenized_ds = ds.map(tokenize_function, batched=True)


In [ ]:

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,    
    per_device_eval_batch_size=32,
    num_train_epochs=1,                
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True                        
)


In [ ]:
def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    probs = p.predictions[:, 1]
    labels = p.label_ids

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    auc = roc_auc_score(labels, probs)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    }

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()


In [ ]:

model.save_pretrained("BASE_DIR/fine_tuned_xlmr")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_xlmr")


In [ ]:
import matplotlib.pyplot as plt


log_history = trainer.state.log_history


train_loss = next(entry["loss"] for entry in log_history if "loss" in entry and entry.get("epoch") == 1.0)
val_loss = next(entry["eval_loss"] for entry in log_history if "eval_loss" in entry and entry.get("epoch") == 1.0)


plt.figure(figsize=(6, 4))
plt.plot([1], [train_loss], marker='o', label="Training Loss")
plt.plot([1], [val_loss], marker='o', label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ========= التقييم الكامل =========
predictions = trainer.predict(tokenized_ds["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]

In [ ]:
# ========= حساب المقاييس =========
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Validation Set")
plt.grid(False)
plt.show()

In [ ]:
# ========= رسم ROC Curve =========
fpr, tpr, _ = roc_curve(y_true, y_probs)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Validation Set")
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
# ========= رسم Bar Plot للمقاييس =========
plt.figure(figsize=(6, 4))
plt.bar(["Accuracy", "Precision", "Recall", "F1", "AUC"], [acc, prec, rec, f1, auc])
plt.ylim(0, 1)
plt.title("Validation Metrics")
plt.ylabel("Score")
plt.grid(axis="y")
plt.show()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import numpy as np


model_path = "BASE_DIR/fine_tuned_xlmr"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


def classify_news(text):

    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)


    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()

    label = "Real" if pred == 1 else "Fake"
    confidence = probs[0][pred].item()
    print(f" Prediction: {label} ({confidence * 100:.2f}%)")


sample_text = "highness prince sheikh nawaf alahmad received message condolence chief national guard highness sheikh salem alali expressed highness sincere condolences death sheikh mansour alahmad praying almighty god deceased covered vast mercy forgiveness highness may inspire patience good condolences highness prince sent reply message highness sheikh salem alali expressed sincere thanks good feelings sincere condolences expressed painful affliction wishing highness grant good health wellness"
classify_news(sample_text)


# **2nd try**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,   
    per_device_eval_batch_size=32,
    num_train_epochs=3,                
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True                         
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()


In [ ]:

model.save_pretrained("BASE_DIR/fine_tuned_xlmr3")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_xlmr3")


In [ ]:
import matplotlib.pyplot as plt


log_history = trainer.state.log_history


train_loss = next(entry["loss"] for entry in log_history if "loss" in entry and entry.get("epoch") == 1.0)
val_loss = next(entry["eval_loss"] for entry in log_history if "eval_loss" in entry and entry.get("epoch") == 1.0)


plt.figure(figsize=(6, 4))
plt.plot([1], [train_loss], marker='o', label="Training Loss")
plt.plot([1], [val_loss], marker='o', label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ========= التقييم الكامل =========
predictions = trainer.predict(tokenized_ds["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]

In [ ]:
# ========= حساب المقاييس =========
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n📊 Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Fake", "Real"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Validation Set")
plt.grid(False)
plt.show()

In [ ]:
# ========= رسم ROC Curve =========
fpr, tpr, _ = roc_curve(y_true, y_probs)
plt.figure()
plt.plot(fpr, tpr, label=f"AUC = {auc:.2f}")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Validation Set")
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
# ========= رسم Bar Plot للمقاييس =========
plt.figure(figsize=(6, 4))
plt.bar(["Accuracy", "Precision", "Recall", "F1", "AUC"], [acc, prec, rec, f1, auc])
plt.ylim(0, 1)
plt.title("Validation Metrics")
plt.ylabel("Score")
plt.grid(axis="y")
plt.show()

# **3Try**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "epoch" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry and "epoch" in entry]
epochs = [entry["epoch"] for entry in log_history if "eval_loss" in entry]


train_losses = train_losses[:len(val_losses)]

# رسم المنحنيات
plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss", marker="o")
plt.plot(epochs, val_losses, label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt


predictions = trainer.predict(tokenized_ds["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]


print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


In [ ]:
# ========= حساب المقاييس =========
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n📊 Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--', label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
model.save_pretrained("BASE_DIR/fine_tuned_xlmr_final2")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_xlmr_final2")


# **4**

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "epoch" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry and "epoch" in entry]
epochs = [entry["epoch"] for entry in log_history if "eval_loss" in entry]


train_losses = train_losses[:len(val_losses)]


plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss", marker="o")
plt.plot(epochs, val_losses, label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ========= حساب المقاييس =========
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n📊 Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
model.save_pretrained("BASE_DIR/fine_tuned_xlmr_final2-16batch")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_xlmr_final2-16batch")


# **4**

In [ ]:
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="steps",
    save_strategy="steps",
    logging_strategy="steps",
    eval_steps=1000,
    save_steps=1000,
    logging_steps=1000,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=4, 
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    save_total_limit=2,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)]  # 🛑 سيتوقف إذا لم يتحسن الأداء بعد تقييمين متتاليين
)


In [ ]:
train_result = trainer.train()


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_steps = [entry["step"] for entry in log_history if "loss" in entry]
train_losses = [entry["loss"] for entry in log_history if "loss" in entry]
val_steps = [entry["step"] for entry in log_history if "eval_loss" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry]

plt.figure(figsize=(8, 5))
plt.plot(train_steps[:len(val_losses)], train_losses[:len(val_losses)], label="Training Loss", marker='o')
plt.plot(val_steps, val_losses, label="Validation Loss", marker='o')
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np


preds = trainer.predict(tokenized_ds["validation"])
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)


print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, auc


y_scores = preds.predictions[:, 1]
fpr, tpr, _ = roc_curve(y_true, y_scores)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
model.save_pretrained("BASE_DIR/fine_tuned_best_model")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_best_model")


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("BASE_DIR/fine_tuned_best_model")
tokenizer = AutoTokenizer.from_pretrained("BASE_DIR/fine_tuned_best_model")


In [ ]:
import shutil

model_dir = "BASE_DIR/fine_tuned_best_model"
output_zip = "/content/fine_tuned_best_model.zip"


shutil.make_archive(base_name=output_zip.replace(".zip", ""), format='zip', root_dir=model_dir)


In [ ]:
from google.colab import files
files.download(output_zip)


# 6

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    #weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "epoch" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry and "epoch" in entry]
epochs = [entry["epoch"] for entry in log_history if "eval_loss" in entry]


train_losses = train_losses[:len(val_losses)]


plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss", marker="o")
plt.plot(epochs, val_losses, label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt


predictions = trainer.predict(tokenized_ds["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]


print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


In [ ]:
# ========= حساب المقاييس =========
acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n📊 Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
model.save_pretrained("BASE_DIR/fine_tuned_best_model_v2")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_best_model_v2")


# 7

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    report_to="none",
    fp16=True
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
train_result = trainer.train()

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [entry["loss"] for entry in log_history if "loss" in entry and "epoch" in entry]
val_losses = [entry["eval_loss"] for entry in log_history if "eval_loss" in entry and "epoch" in entry]
epochs = [entry["epoch"] for entry in log_history if "eval_loss" in entry]


train_losses = train_losses[:len(val_losses)]


plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss", marker="o")
plt.plot(epochs, val_losses, label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

predictions = trainer.predict(tokenized_ds["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]


print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))


In [ ]:

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_probs)

print("\n Metrics on Validation Set:")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


predictions = trainer.predict(tokenized_ds["test"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)
y_probs = predictions.predictions[:, 1]  # احتمالية "Real" (label 1)


print("\n Classification Report on Test Set:")
print(classification_report(y_true, y_pred, target_names=["Fake", "Real"]))

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc_score = roc_auc_score(y_true, y_probs)

print("\n Metrics on Test Set:")
print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"AUC       : {auc_score:.4f}")


cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (Test Set)")
plt.tight_layout()
plt.show()


fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve (Test Set)')
plt.legend(loc="lower right")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
model.save_pretrained("BASE_DIR/fine_tuned_best_model_v3")
tokenizer.save_pretrained("BASE_DIR/fine_tuned_best_model_v3")
